In [2]:
import os
import tqdm
import numpy as np
import pandas as pd
from tushare import pro_api, set_token
from datetime import datetime
from dateutil.relativedelta import relativedelta
import time

In [3]:
#set_token('b549c6e18f71105519447d872727cc2b7f0022f071c9eb27d0256ebb')
token = 'f8e3549464e9d28a0f272ba4e6308a873fe195f6415c20dbe40def91e3c0'
pro = pro_api(token)
pro._DataApi__token = token
pro._DataApi__http_url = 'http://lianghua.nanyangqiankun.top'

### 交易日历

In [10]:
#trade_cal = pro.trade_cal()
#trade_cal.to_csv('data/trade_cal.csv', index=False)
trade_cal = pd.read_csv('data/trade_cal.csv', dtype={'cal_date': str})
data_cal = trade_cal[(trade_cal['cal_date']>='20260101')&(trade_cal['cal_date']<'20260401')]

### 日线

In [4]:
data_path = 'data/daily_K/'
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        daily = pro.daily(trade_date=d['cal_date'])
        adj_factor = pro.adj_factor(trade_date=d['cal_date'])
        price = pd.merge(daily, adj_factor)
        price = price.drop(columns=['change', 'pct_chg']).sort_values('ts_code').reset_index(drop=True)
        price['trade_date'] = pd.to_datetime(price['trade_date'])
        price.to_feather(os.path.join(data_path, f'stock-{d["cal_date"]}.ftr'))
    else:
        continue

90it [01:02,  1.43it/s]


### 每日基础数据

In [5]:
data_path = 'data/daily_basic/'
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        daily_basic = pro.daily_basic(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
        daily_basic['trade_date'] = pd.to_datetime(daily_basic['trade_date'])
        daily_basic.to_feather(os.path.join(data_path, f'basic-{d["cal_date"]}.ftr'))
    else:
        continue

90it [01:23,  1.08it/s]


### 股票列表

In [ ]:
# industry = pro.index_member_all()
# industry.to_csv('data/industry.csv', index=False)
# allstock = pro.stock_basic()
# allstock.to_csv('data/allstock.csv', index=False)

In [ ]:
allstock = pd.read_csv('data/allstock.csv')
allstock_list = allstock.ts_code.tolist()

### 财务指标

In [ ]:
# fins = []
# batch_size = 200
# with tqdm.tqdm(total=len(allstock_list), desc="处理股票", unit="stock") as pbar:
#     for i in range(0, len(allstock_list), batch_size):
#         batch = allstock_list[i:i + batch_size]
#         for d in batch:
#             tmp = pro.fina_indicator(ts_code=d, start_date='20050930', end_date='20130929').drop_duplicates(subset=['ts_code', 'end_date'], keep='first')
#             fins.append(tmp)
#             pbar.update(1)
        
#         # 每批次结束后休息
#         if i + batch_size < len(allstock_list):
#             time.sleep(35)
# df_fins = pd.concat(fins)
# for d in df_fins.end_date.unique():
#     df_fins[df_fins['end_date'] == d].to_feather(os.path.join('data/finance/', f'fina-{d}.ftr'))

处理股票: 100%|██████████| 2286/2286 [11:46<00:00,  3.23stock/s]  


In [27]:
for d in pd.date_range('2009-12-31', '2025-12-31', freq='QE').strftime('%Y%m%d').tolist():
    tmp1 = pro.balancesheet_vip(period=d).drop_duplicates(subset=['ts_code', 'end_date'], keep='last')
    tmp2 = pro.cashflow_vip(period=d).drop_duplicates(subset=['ts_code', 'end_date'], keep='last').drop(columns=['ann_date','f_ann_date','comp_type','end_type','report_type','update_flag'])
    tmp3 = pro.income_vip(period=d).drop_duplicates(subset=['ts_code', 'end_date'], keep='last').drop(columns=['ann_date','f_ann_date','comp_type','end_type','report_type','update_flag'])
    sheet = pd.merge(tmp1, tmp2, on=['ts_code', 'end_date'], how='outer').merge(tmp3, on=['ts_code', 'end_date'], how='outer')
    sheet.to_feather(os.path.join('data/finance/sheet', f'sheet-{d}.ftr'))

### 资金流向&龙虎榜&融资融券

In [7]:
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        moneyflow = pro.moneyflow(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
        moneyflow['trade_date'] = pd.to_datetime(moneyflow['trade_date'])
        moneyflow.to_feather(os.path.join('data/moneyflow/', f'moneyflow-{d["cal_date"]}.ftr'))
        toplist = pro.top_list(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
        toplist['trade_date'] = pd.to_datetime(toplist['trade_date'])
        toplist.to_feather(os.path.join('data/toplist/', f'toplist-{d["cal_date"]}.ftr'))
        tmp = pro.margin_detail(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
        tmp['trade_date'] = pd.to_datetime(tmp['trade_date'])
        tmp.to_feather(os.path.join('data/rzrq/', f'margin-{d["cal_date"]}.ftr'))
    else:
        continue

90it [02:11,  1.46s/it]


### 宏观

In [8]:
cpi = pro.cn_cpi(start_m='200512', end_m='202603')
ppi = pro.cn_ppi(start_m='200512', end_m='202603')
money = pro.cn_m(start_m='200512', end_m='202603')
sf = pro.sf_month(start_m='200512', end_m='202603')

df_macro = cpi.merge(ppi, on='month', how='left').merge(money, on='month', how='left').merge(sf, on='month', how='left')
df_macro['month'] = pd.to_datetime(df_macro['month'], format='%Y%m')
df_macro.to_csv(os.path.join('data/', 'macro.csv'), index=False)

### 指数相关

In [4]:
index_list = [
    '000300.SH',   # 沪深300
    '000001.SH',     # 上证指数
    '399001.SZ',     # 深证成指
    '000905.SH',   # 中证500
    '399006.SZ',   # 创业板指
    '000016.SH',   # 上证50
    '000688.SH',   # 科创50
    '000852.SH',   # 中证1000
]

for d in tqdm.tqdm(index_list):
    index = pro.index_daily(ts_code=d, start_date='20051230', end_date='20260331')
    index['trade_date'] = pd.to_datetime(index['trade_date'])
    index.to_csv(os.path.join('data/index/index_daily_K/', f'{d}.csv'), index=False)
    indexbasic = pro.index_dailybasic(ts_code=d, start_date='20051230', end_date='20260331')
    indexbasic['trade_date'] = pd.to_datetime(indexbasic['trade_date'])
    indexbasic.to_csv(os.path.join('data/index/index_daily_basic/', f'{d}_basic.csv'), index=False)


100%|██████████| 8/8 [00:12<00:00,  1.52s/it]


### 股东增减持

In [5]:
current = datetime.strptime('20260101', '%Y%m%d')
with tqdm.tqdm() as pbar:
    while current <= datetime.strptime('20260331', '%Y%m%d'):
        # 当月第一天
        month_start = current.replace(day=1)
        # 当月最后一天
        month_end = (month_start + relativedelta(months=1) - relativedelta(days=1))
        df_hodlers =  pro.stk_holdertrade(start_date = month_start.strftime('%Y%m%d'), end_date = month_end.strftime('%Y%m%d'))
        if len(df_hodlers)>=3000:
            print(f"Warning: {month_start.strftime('%Y-%m')} may exceed API limits.")
        else:
            df_hodlers.to_feather(os.path.join('data/holdertrade/', f'holders-{month_start.strftime("%Y%m")}.ftr'))
        pbar.update(1)
        # 进入下一月
        current = month_start + relativedelta(months=1)


3it [00:00,  3.18it/s]


### 分析师预测

In [7]:
report = pd.DataFrame()
for d in tqdm.tqdm(pd.date_range('2026-01-01', '2026-03-31').strftime('%Y%m%d').tolist()):
    tmp = pro.report_rc(report_date=d)
    if tmp.empty:
        continue
    else:
        report = pd.concat([report, tmp])
for i in report.quarter.unique():
    if not os.path.exists(os.path.join('data/report_rc', f'report-{i}.ftr')):
        report[report['quarter'] == i].to_feather(os.path.join('data/report_rc', f'report-{i}.ftr'))
    else:
        datas = pd.read_feather(os.path.join('data/report_rc', f'report-{i}.ftr'))
        datas = pd.concat([datas, report[report['quarter'] == i]])
        datas.to_feather(os.path.join('data/report_rc', f'report-{i}.ftr'))

100%|██████████| 90/90 [00:22<00:00,  3.97it/s]


### 筹码胜率

In [11]:
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        tmp = pro.cyq_perf(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
        tmp['trade_date'] = pd.to_datetime(tmp['trade_date'])
        if tmp.empty:
            continue
        tmp.to_feather(os.path.join('data/cyq/', f'cyq-{d["cal_date"]}.ftr'))
    else:
        continue

90it [00:40,  2.21it/s]


### ETF

In [12]:
data_path = 'data/ETF/'
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        daily = pro.fund_daily(trade_date=d['cal_date'])
        adj_factor = pro.fund_adj(trade_date=d['cal_date'])
        price = pd.merge(daily, adj_factor)
        price = price.drop(columns=['change', 'pct_chg']).sort_values('ts_code').reset_index(drop=True)
        price['trade_date'] = pd.to_datetime(price['trade_date'])
        price.to_feather(os.path.join(data_path, f'etf-{d["cal_date"]}.ftr'))
    else:
        continue

90it [00:40,  2.21it/s]


In [29]:
data_path = 'data/ETF/'
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        price = pd.read_feather(os.path.join(data_path, f'etf-{d["cal_date"]}.ftr'))
        daily_basic = pro.etf_share_size(trade_date=d['cal_date'],fields='trade_date, ts_code, total_share, total_size, nav')
        daily_basic['trade_date'] = pd.to_datetime(daily_basic['trade_date'])
        price = price.merge(daily_basic, on=['ts_code', 'trade_date'], how='outer')
        price.to_feather(os.path.join(data_path, f'etf-{d["cal_date"]}.ftr'))
    else:
        continue

7307it [19:34,  6.22it/s]


In [13]:
df = pro.etf_basic()
df['list_date'] = pd.to_datetime(df['list_date'])
df.to_csv('data/alletf.csv', index=False)

### 期货

In [14]:
data_path = 'data/future/'
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        daily = pro.fut_daily(trade_date=d['cal_date'])
        daily = daily.sort_values('ts_code').reset_index(drop=True)
        daily['trade_date'] = pd.to_datetime(daily['trade_date'])
        daily.to_feather(os.path.join(data_path, f'future-{d["cal_date"]}.ftr'))
    else:
        continue

90it [00:21,  4.20it/s]


### 月度金股

In [15]:
for i in pd.date_range('2026-01-01', '2026-03-31', freq='MS').strftime('%Y%m').tolist():
    df = pro.broker_recommend(month=i)
    df = df.groupby(['ts_code','month']).size().reset_index(name='count')
    df.to_csv(os.path.join('data/month_recommend/', f'recommend-{i}.csv'), index=False)

### 打板相关

In [16]:
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        daily = pro.limit_list_d(trade_date=d["cal_date"])
        daily['trade_date'] = pd.to_datetime(daily['trade_date'])
        daily.to_feather(os.path.join('data/limit_list/', f'limit_list-{d["cal_date"]}.ftr'))
    else:
        continue

90it [00:11,  7.86it/s]


### 其他